In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage
from typing import TypedDict, Literal, Annotated
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator
import os
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
load_dotenv()

True

In [3]:
llm = ChatOpenAI(
    model="openai/gpt-oss-120b",
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY")
)

In [4]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [5]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [6]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [7]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [8]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the slice of pizza start a band?\n\nBecause it already had the perfect *crust* and a lot of *cheese* for the fans! 🍕😄',
 'explanation': '### The joke\n\n> **Why did the slice of pizza start a band?**  \n> **Because it already had the perfect *crust* and a lot of *cheese* for the fans!** 🍕😄  \n\n### Why it works – a step‑by‑step breakdown\n\n| Pizza element | What it means in the pizza world | What it hints at in the music world (the “punch line”) | Why the connection is funny |\n|---------------|----------------------------------|--------------------------------------------------------|-----------------------------|\n| **Crust**     | The outer, baked edge that holds the toppings together. | The **foundation** or **“base”** of a band – the rhythm section, the beat, the structural “crust” that everything else sits on. | We don’t normally talk about a band having a “crust,” so the word is taken literally from pizza and then repurposed as a musical met

In [9]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the slice of pizza start a band?\n\nBecause it already had the perfect *crust* and a lot of *cheese* for the fans! 🍕😄', 'explanation': '### The joke\n\n> **Why did the slice of pizza start a band?**  \n> **Because it already had the perfect *crust* and a lot of *cheese* for the fans!** 🍕😄  \n\n### Why it works – a step‑by‑step breakdown\n\n| Pizza element | What it means in the pizza world | What it hints at in the music world (the “punch line”) | Why the connection is funny |\n|---------------|----------------------------------|--------------------------------------------------------|-----------------------------|\n| **Crust**     | The outer, baked edge that holds the toppings together. | The **foundation** or **“base”** of a band – the rhythm section, the beat, the structural “crust” that everything else sits on. | We don’t normally talk about a band having a “crust,” so the word is taken literally from pizza and then repurpos

In [10]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the slice of pizza start a band?\n\nBecause it already had the perfect *crust* and a lot of *cheese* for the fans! 🍕😄', 'explanation': '### The joke\n\n> **Why did the slice of pizza start a band?**  \n> **Because it already had the perfect *crust* and a lot of *cheese* for the fans!** 🍕😄  \n\n### Why it works – a step‑by‑step breakdown\n\n| Pizza element | What it means in the pizza world | What it hints at in the music world (the “punch line”) | Why the connection is funny |\n|---------------|----------------------------------|--------------------------------------------------------|-----------------------------|\n| **Crust**     | The outer, baked edge that holds the toppings together. | The **foundation** or **“base”** of a band – the rhythm section, the beat, the structural “crust” that everything else sits on. | We don’t normally talk about a band having a “crust,” so the word is taken literally from pizza and then repurpo

In [11]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti get a promotion?\n\nBecause it *always* knows how to *handle* a *pasta*-ble situation! 🍝😄',
 'explanation': '**Explanation of the Joke**\n\n> *Why did the spaghetti get a promotion?  \n> Because it **always** knows how to **handle** a **pasta‑ble** situation! 🍝😄*\n\n---\n\n### 1. The Set‑up  \nThe joke starts with a classic “Why did…?” question. In everyday conversation, that format is used to set up a punchline that explains an unexpected or humorous reason for something. Here the subject is *spaghetti*, an inanimate type of pasta, which immediately signals that the answer will involve wordplay rather than a realistic scenario.\n\n### 2. The Punchline – The Wordplay  \n\n| Word in the punchline | What it usually means | How it’s twisted in the joke |\n|-----------------------|----------------------|------------------------------|\n| **always**            | “all the time,” a trait of reliability | Suggests the spaghetti is *consistentl

In [12]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the slice of pizza start a band?\n\nBecause it already had the perfect *crust* and a lot of *cheese* for the fans! 🍕😄', 'explanation': '### The joke\n\n> **Why did the slice of pizza start a band?**  \n> **Because it already had the perfect *crust* and a lot of *cheese* for the fans!** 🍕😄  \n\n### Why it works – a step‑by‑step breakdown\n\n| Pizza element | What it means in the pizza world | What it hints at in the music world (the “punch line”) | Why the connection is funny |\n|---------------|----------------------------------|--------------------------------------------------------|-----------------------------|\n| **Crust**     | The outer, baked edge that holds the toppings together. | The **foundation** or **“base”** of a band – the rhythm section, the beat, the structural “crust” that everything else sits on. | We don’t normally talk about a band having a “crust,” so the word is taken literally from pizza and then repurpos

In [13]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the slice of pizza start a band?\n\nBecause it already had the perfect *crust* and a lot of *cheese* for the fans! 🍕😄', 'explanation': '### The joke\n\n> **Why did the slice of pizza start a band?**  \n> **Because it already had the perfect *crust* and a lot of *cheese* for the fans!** 🍕😄  \n\n### Why it works – a step‑by‑step breakdown\n\n| Pizza element | What it means in the pizza world | What it hints at in the music world (the “punch line”) | Why the connection is funny |\n|---------------|----------------------------------|--------------------------------------------------------|-----------------------------|\n| **Crust**     | The outer, baked edge that holds the toppings together. | The **foundation** or **“base”** of a band – the rhythm section, the beat, the structural “crust” that everything else sits on. | We don’t normally talk about a band having a “crust,” so the word is taken literally from pizza and then repurpo

Viaje en el tiempo

In [17]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f11d513-cec7-6eed-8000-ad99dffbf43d"}})

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f11d513-cec7-6eed-8000-ad99dffbf43d'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-03-11T13:50:17.688437+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f11d513-cec6-6106-bfff-ca147de58dc5'}}, tasks=(PregelTask(id='6d009f1c-f127-cc5b-dbb2-251daf1c7913', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the slice of pizza start a band?\n\nBecause it already had the perfect *crust* and a lot of *cheese* for the fans! 🍕😄'}),), interrupts=())

In [18]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f11d513-cec7-6eed-8000-ad99dffbf43d"}})

{'topic': 'pizza',
 'joke': 'Why did the pizza apply for a job?\n\nBecause it heard the company was looking for people who could *deliver* under pressure and had a lot of “topping” potential! 🍕😄',
 'explanation': '**Explanation of the Joke**\n\n> **Why did the pizza apply for a job?**  \n> **Because it heard the company was looking for people who could *deliver* under pressure and had a lot of “topping” potential!** 🍕😄\n\n---\n\n### 1. The Core Structure: A Classic “Why‑did‑the‑X‑do‑Y?” Setup\n- The joke follows the familiar pattern “Why did the X …?” which primes the listener for a punch‑line that plays on words related to the subject (in this case, pizza).\n\n### 2. The Two Main Puns\n\n| Phrase in the punch‑line | Literal meaning (pizza’s world) | Figurative meaning (job‑search world) |\n|--------------------------|--------------------------------|----------------------------------------|\n| **“deliver”** | A pizza’s primary job is to be delivered to customers. | In business, *deliv

In [19]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it heard the company was looking for people who could *deliver* under pressure and had a lot of “topping” potential! 🍕😄', 'explanation': '**Explanation of the Joke**\n\n> **Why did the pizza apply for a job?**  \n> **Because it heard the company was looking for people who could *deliver* under pressure and had a lot of “topping” potential!** 🍕😄\n\n---\n\n### 1. The Core Structure: A Classic “Why‑did‑the‑X‑do‑Y?” Setup\n- The joke follows the familiar pattern “Why did the X …?” which primes the listener for a punch‑line that plays on words related to the subject (in this case, pizza).\n\n### 2. The Two Main Puns\n\n| Phrase in the punch‑line | Literal meaning (pizza’s world) | Figurative meaning (job‑search world) |\n|--------------------------|--------------------------------|----------------------------------------|\n| **“deliver”** | A pizza’s primary job is to be delivered to customers. |

Actualizar estado

In [20]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f11d513-cec6-6106-bfff-ca147de58dc5", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f11d523-10b3-6873-8000-716fa229bbf4'}}

In [21]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f11d523-10b3-6873-8000-716fa229bbf4'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-03-11T13:57:07.253862+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f11d513-cec6-6106-bfff-ca147de58dc5'}}, tasks=(PregelTask(id='13a59a02-193c-4776-98d3-2ee89fbaf534', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it heard the company was looking for people who could *deliver* under pressure and had a lot of “topping” potential! 🍕😄', 'explanation': '**Explanation of the Joke**\n\n> **Why did the pizza apply for a job?**  \n> **Because it heard the company was looking for people who could *deli

In [22]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f11d523-10b3-6873-8000-716fa229bbf4"}})

{'topic': 'samosa',
 'joke': 'Why did the samosa go to therapy?  \n\nBecause every time it tried to open up, it just kept getting stuffed with too many feelings!',
 'explanation': '**Joke:**  \n*Why did the samosa go to therapy?*  \n*Because every time it tried to open up, it just kept getting stuffed with too many feelings!*\n\n---\n\n## Why It’s Funny – A Step‑by‑Step Breakdown  \n\n| Element of the Joke | What It Means Literally | What It Means Figuratively (the “punchline”) |\n|---------------------|------------------------|-----------------------------------------------|\n| **Samosa** | A deep‑fried pastry that is **filled** (stuffed) with a mixture of spiced potatoes, peas, meat, etc. | The samosa becomes a stand‑in for a person who is “filled” with emotions. |\n| **Go to therapy** | A person visits a mental‑health professional to talk about problems. | Implies the samosa (as a person) has emotional issues that need professional help. |\n| **Open up** | To reveal the interior of 

In [23]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'Why did the samosa go to therapy?  \n\nBecause every time it tried to open up, it just kept getting stuffed with too many feelings!', 'explanation': '**Joke:**  \n*Why did the samosa go to therapy?*  \n*Because every time it tried to open up, it just kept getting stuffed with too many feelings!*\n\n---\n\n## Why It’s Funny – A Step‑by‑Step Breakdown  \n\n| Element of the Joke | What It Means Literally | What It Means Figuratively (the “punchline”) |\n|---------------------|------------------------|-----------------------------------------------|\n| **Samosa** | A deep‑fried pastry that is **filled** (stuffed) with a mixture of spiced potatoes, peas, meat, etc. | The samosa becomes a stand‑in for a person who is “filled” with emotions. |\n| **Go to therapy** | A person visits a mental‑health professional to talk about problems. | Implies the samosa (as a person) has emotional issues that need professional help. |\n| **Open up** | To rev